# Module 3 CSO overview and provisional CSO-to-MPS spatial mapping

This notebook reads CSO locations, reads Eksel/Overpelt CSO time series, summarizes data availability and zero-flow rows, and creates provisional straight-line distance tables to MPS coordinate candidates.

In [ ]:
"""
Module 3 CSO overview and provisional CSO-to-MPS spatial mapping.

This script is intentionally written in a simple notebook style.
It does four things:
1. Read CSO discharge-point locations from Lozingspunten.
2. Read CSO time-series rows for Eksel and Overpelt.
3. Summarise data availability and zero-flow rows per physical CSO point.
4. Create provisional straight-line distance tables from CSO points to candidate MPS locations.

Important limitation:
The exact 2017 MP01-MP07 measurement coordinates are not confirmed yet.
Therefore, all CSO-to-MPS mappings here are provisional candidate mappings.
"""

from pathlib import Path
from zipfile import ZipFile
from datetime import datetime
import re
import xml.etree.ElementTree as ET

import pandas as pd
import geopandas as gpd

## Cell 1 - File paths

In [ ]:
DATA_DIR = Path('/mnt/data')

LOZINGEN_FILE = DATA_DIR / 'Lozingspunten (1).xlsx'
EK_FILE = DATA_DIR / 'EK files samengevoegd.xlsx'
OVE_FILE = DATA_DIR / 'OVE files samengevoegd.xlsx'
MPS_GPKG_FILE = DATA_DIR / 'MPS_stations.gpkg'

OUTPUT_WORKBOOK = DATA_DIR / 'M5_Module3_CSO_overview_and_spatial_mapping.xlsx'
OUTPUT_POINT_SUMMARY = DATA_DIR / 'M5_Module3_CSO_point_summary.csv'
OUTPUT_MP_DISTANCE = DATA_DIR / 'M5_Module3_CSO_to_MP_candidate_distances.csv'

for path in [LOZINGEN_FILE, EK_FILE, OVE_FILE, MPS_GPKG_FILE]:
    if not path.exists():
        raise FileNotFoundError(f'Missing input file: {path}')

## Cell 2 - Simple helper functions for IDs

In [ ]:
def split_source_ids(value):
    """Split source ID cells that contain multiple IDs separated by / or newlines."""
    if pd.isna(value):
        return []

    parts = re.split(r'/|\n', str(value))
    clean_parts = []

    for part in parts:
        part = part.strip()
        if part:
            clean_parts.append(part)

    return clean_parts


def make_match_key(value):
    """
    Create a stable key for matching CSO IDs between location and time-series files.

    The files contain small naming differences, for example:
    - EK_1168_2 versus EK1168_2
    - OVE_114_O1ov_1 versus OVE_114_01ov_1
    - 0VE versus OVE
    """
    if pd.isna(value):
        return ''

    text = str(value).strip().upper()

    text = text.replace('0VE', 'OVE')
    text = text.replace('O1OV', '01OV')
    text = text.replace(',', '')

    # Remove one trailing technical suffix where this is how the files differ.
    # Example: EK_1128_1 -> EK_1128
    if text.count('_') >= 2 or re.match(r'^[A-Z]+[0-9][0-9A-Z]*_\d+$', text):
        text = re.sub(r'_\d+$', '', text)

    # Known small typo/format difference in the Overpelt IDs.
    text = text.replace('K000010', 'K00010')

    # Keep only letters and numbers.
    text = re.sub(r'[^A-Z0-9]', '', text)

    return text

## Cell 3 - Read CSO discharge-point locations

In [ ]:
def read_cso_locations(path):
    """Read and clean the CSO sheet from Lozingspunten."""
    locations = pd.read_excel(path, sheet_name='CSO')

    locations = locations.rename(columns={
        'xlsx-bestandsnaam': 'source_id_raw',
        'Naam': 'source_name',
        'Zuiv.gebied': 'site',
        'x(m)': 'x_m',
        'y(m)': 'y_m',
        'Ontvangende waterloop': 'receiving_watercourse',
        'Deelbekken': 'subbasin',
        'Koppeling in ICM ? ': 'coupled_in_icm',
        'Opmerkingen': 'remarks',
    })

    # We currently only have CSO time-series files for Eksel and Overpelt.
    locations = locations[locations['site'].isin(['Eksel', 'Overpelt'])].copy()

    # Exclude RWZI overflow points from the CSO discharge-point overview.
    # This gives 10 Eksel + 25 Overpelt points.
    locations = locations[
        ~locations['source_name'].str.contains('RWZI', case=False, na=False)
    ].copy()

    locations['x_m'] = pd.to_numeric(locations['x_m'], errors='coerce')
    locations['y_m'] = pd.to_numeric(locations['y_m'], errors='coerce')

    return locations.reset_index(drop=True)


cso_locations = read_cso_locations(LOZINGEN_FILE)

print('CSO discharge points from Lozingspunten:')
print(cso_locations['site'].value_counts().to_string())

## Cell 4 - Expand physical points to source IDs

In [ ]:
def build_point_to_source_table(locations):
    """Create one row per expected source ID for each physical discharge point."""
    rows = []

    for _, point in locations.iterrows():
        source_ids = split_source_ids(point['source_id_raw'])

        for source_id in source_ids:
            rows.append({
                'site': point['site'],
                'physical_point_id': point['site'] + '_' + str(point['source_name']),
                'source_name': point['source_name'],
                'expected_source_id': source_id,
                'match_key': make_match_key(source_id),
                'x_m': point['x_m'],
                'y_m': point['y_m'],
                'receiving_watercourse': point['receiving_watercourse'],
                'subbasin': point['subbasin'],
                'coupled_in_icm': point['coupled_in_icm'],
                'remarks': point['remarks'],
            })

    return pd.DataFrame(rows)


point_to_source = build_point_to_source_table(cso_locations)

print('\nExpected source IDs after splitting combined cells:')
print(point_to_source['site'].value_counts().to_string())

## Cell 5 - Helpers for reading only useful Excel cells from large CSO workbooks

In [ ]:
MAIN_NS = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
REL_NS = '{http://schemas.openxmlformats.org/officeDocument/2006/relationships}'

EXCEL_ZERO_DATE = datetime(1899, 12, 30)
START_2017_SERIAL = (datetime(2017, 1, 1) - EXCEL_ZERO_DATE).days
END_2017_SERIAL = (datetime(2018, 1, 1) - EXCEL_ZERO_DATE).days

VALUE_COLUMNS = ['Q', 'BOD', 'COD', 'ZS', 'KJN', 'NH4', 'NO3', 'NTOT', 'PTOT']


def read_shared_strings(xlsx_file):
    """Read text values stored by Excel inside the .xlsx file."""
    with ZipFile(xlsx_file) as zip_file:
        if 'xl/sharedStrings.xml' not in zip_file.namelist():
            return []
        xml_text = zip_file.read('xl/sharedStrings.xml')

    root = ET.fromstring(xml_text)
    strings = []

    for item in root:
        strings.append(''.join(item.itertext()))

    return strings


def find_sheet_xml_path(xlsx_file, sheet_name):
    """Find the internal XML path for a worksheet name."""
    with ZipFile(xlsx_file) as zip_file:
        workbook_xml = ET.fromstring(zip_file.read('xl/workbook.xml'))
        rels_xml = ET.fromstring(zip_file.read('xl/_rels/workbook.xml.rels'))

    rel_targets = {}
    for relationship in rels_xml:
        rel_targets[relationship.attrib['Id']] = relationship.attrib['Target']

    sheets = workbook_xml.find(MAIN_NS + 'sheets')

    for sheet in sheets:
        if sheet.attrib['name'] == sheet_name:
            rel_id = sheet.attrib[REL_NS + 'id']
            return 'xl/' + rel_targets[rel_id]

    raise ValueError(f'Sheet not found: {sheet_name}')


def get_cell_text(cell, shared_strings):
    """Return the readable value of one Excel cell."""
    value_node = cell.find(MAIN_NS + 'v')

    if value_node is None:
        return None

    text = value_node.text

    if cell.attrib.get('t') == 's':
        return shared_strings[int(text)]

    return text


def read_cso_workbook(xlsx_file, sheet_name, site):
    """
    Read CSO time-series rows for 2017.

    We keep the raw Excel serial time. It is enough for grouping and avoids
    timezone confusion. It can be converted later if needed.
    """
    print(f'\nReading CSO time series: {site} / {xlsx_file.name}')

    shared_strings = read_shared_strings(xlsx_file)
    sheet_xml_path = find_sheet_xml_path(xlsx_file, sheet_name)

    rows = []

    with ZipFile(xlsx_file) as zip_file:
        sheet_file = zip_file.open(sheet_xml_path)

        for _, row_element in ET.iterparse(sheet_file, events=('end',)):
            if row_element.tag != MAIN_NS + 'row':
                continue

            row_number = int(row_element.attrib.get('r', '0'))

            # Row 1 is header, row 2 is units.
            if row_number < 3:
                row_element.clear()
                continue

            source_id = None
            time_serial = None
            values = [0.0] * len(VALUE_COLUMNS)

            for cell in row_element:
                if cell.tag != MAIN_NS + 'c':
                    continue

                cell_reference = cell.attrib.get('r', '')
                if not cell_reference:
                    continue

                column = cell_reference[0]
                text = get_cell_text(cell, shared_strings)

                if text is None:
                    continue

                if column == 'A':
                    source_id = str(text).strip().replace(',', '')

                elif column == 'B':
                    try:
                        time_serial = float(text)
                    except ValueError:
                        time_serial = None

                elif column in 'CDEFGHIJK':
                    column_index = ord(column) - ord('C')
                    values[column_index] = float(text)

            row_element.clear()

            if source_id is None or time_serial is None:
                continue

            if time_serial < START_2017_SERIAL or time_serial >= END_2017_SERIAL:
                continue

            row = {
                'site': site,
                'timeseries_source_id': source_id,
                'match_key': make_match_key(source_id),
                'time_excel_serial': time_serial,
                'source_file': xlsx_file.name,
                'source_sheet': sheet_name,
            }

            for name, value in zip(VALUE_COLUMNS, values):
                row[name] = value

            rows.append(row)

    return pd.DataFrame(rows)

## Cell 6 - Read both CSO time-series files

In [ ]:
eksel_data = read_cso_workbook(EK_FILE, 'alle data', 'Eksel')
overpelt_data = read_cso_workbook(OVE_FILE, 'Blad1', 'Overpelt')

cso_data = pd.concat([eksel_data, overpelt_data], ignore_index=True)

print('\nCSO time-series rows loaded:')
print(cso_data['site'].value_counts().to_string())
print('Total rows:', len(cso_data))

## Cell 7 - Match time series to physical discharge points

In [ ]:
joined = cso_data.merge(
    point_to_source[
        ['site', 'physical_point_id', 'source_name', 'expected_source_id', 'match_key']
    ],
    on=['site', 'match_key'],
    how='inner',
)

print('\nRows matched to non-RWZI physical CSO points:')
print(joined['site'].value_counts().to_string())
print('Total matched rows:', len(joined))

## Cell 8 - Aggregate to physical point and time

In [ ]:
point_time = (
    joined
    .groupby(
        ['site', 'physical_point_id', 'source_name', 'time_excel_serial'],
        as_index=False,
    )[VALUE_COLUMNS]
    .sum()
)

point_time['zero_flow_row'] = point_time['Q'].eq(0)
point_time['all_zero_row'] = point_time[VALUE_COLUMNS].eq(0).all(axis=1)
point_time['active_flow_row'] = point_time['Q'].gt(0)
point_time['volume_m3_15min'] = point_time['Q'] * 900

print('\nRows after grouping to physical point and time:', len(point_time))

## Cell 9 - Summarise CSO data per physical discharge point

In [ ]:
point_summary = (
    point_time
    .groupby(['site', 'physical_point_id', 'source_name'], as_index=False)
    .agg(
        rows_total=('time_excel_serial', 'count'),
        zero_flow_rows=('zero_flow_row', 'sum'),
        all_zero_rows=('all_zero_row', 'sum'),
        active_flow_rows=('active_flow_row', 'sum'),
        total_volume_m3_2017=('volume_m3_15min', 'sum'),
        max_q_m3s_2017=('Q', 'max'),
    )
)

point_summary['percent_zero_flow_rows'] = (
    100 * point_summary['zero_flow_rows'] / point_summary['rows_total']
)

point_summary['percent_all_zero_rows'] = (
    100 * point_summary['all_zero_rows'] / point_summary['rows_total']
)

expected_by_point = (
    point_to_source
    .groupby(['site', 'physical_point_id', 'source_name'], as_index=False)
    .agg(
        expected_source_ids=('expected_source_id', lambda values: ', '.join(values)),
        expected_source_count=('expected_source_id', 'nunique'),
        x_m=('x_m', 'first'),
        y_m=('y_m', 'first'),
        receiving_watercourse=('receiving_watercourse', 'first'),
        subbasin=('subbasin', 'first'),
        coupled_in_icm=('coupled_in_icm', 'first'),
        remarks=('remarks', 'first'),
    )
)

matched_by_point = (
    joined
    .groupby(['site', 'physical_point_id', 'source_name'], as_index=False)
    .agg(
        matched_timeseries_ids=(
            'timeseries_source_id',
            lambda values: ', '.join(sorted(set(values))),
        ),
        matched_timeseries_count=('timeseries_source_id', 'nunique'),
    )
)

point_summary = expected_by_point.merge(
    matched_by_point,
    on=['site', 'physical_point_id', 'source_name'],
    how='left',
).merge(
    point_summary,
    on=['site', 'physical_point_id', 'source_name'],
    how='left',
)

# Fill values for points without matched data.
point_summary['matched_timeseries_ids'] = point_summary['matched_timeseries_ids'].fillna('')
point_summary['matched_timeseries_count'] = point_summary['matched_timeseries_count'].fillna(0).astype(int)

count_columns = ['rows_total', 'zero_flow_rows', 'all_zero_rows', 'active_flow_rows']
for column in count_columns:
    point_summary[column] = point_summary[column].fillna(0).astype(int)

value_columns = [
    'total_volume_m3_2017', 'max_q_m3s_2017',
    'percent_zero_flow_rows', 'percent_all_zero_rows',
]
for column in value_columns:
    point_summary[column] = point_summary[column].fillna(0.0)

point_summary['has_data_for_all_expected_ids'] = (
    point_summary['matched_timeseries_count'] == point_summary['expected_source_count']
)
point_summary['has_any_flow_data'] = point_summary['rows_total'] > 0
point_summary['has_any_emission_2017'] = point_summary['active_flow_rows'] > 0

point_summary = point_summary.sort_values(['site', 'source_name']).reset_index(drop=True)

print('\nPhysical CSO points in overview:')
print(point_summary['site'].value_counts().to_string())

## Cell 10 - Site-level CSO overview

In [ ]:
site_summary = (
    point_summary
    .groupby('site', as_index=False)
    .agg(
        physical_discharge_points=('source_name', 'count'),
        points_with_all_expected_data=('has_data_for_all_expected_ids', 'sum'),
        points_with_any_flow_data=('has_any_flow_data', 'sum'),
        points_with_emission_2017=('has_any_emission_2017', 'sum'),
        total_rows=('rows_total', 'sum'),
        zero_flow_rows=('zero_flow_rows', 'sum'),
        active_flow_rows=('active_flow_rows', 'sum'),
        total_volume_m3_2017=('total_volume_m3_2017', 'sum'),
        max_q_m3s_2017=('max_q_m3s_2017', 'max'),
    )
)

site_summary['percent_zero_flow_rows'] = (
    100 * site_summary['zero_flow_rows'] / site_summary['total_rows']
)

print('\nSite summary:')
print(site_summary.round(3).to_string(index=False))

incomplete_points = point_summary[
    ~point_summary['has_data_for_all_expected_ids']
].copy()

print('\nPoints without all expected time-series IDs:')
if len(incomplete_points) == 0:
    print('None')
else:
    print(
        incomplete_points[
            ['site', 'source_name', 'expected_source_ids', 'matched_timeseries_ids']
        ].to_string(index=False)
    )

## Cell 11 - Read candidate MPS coordinates

In [ ]:
def read_project_mps_candidates(gpkg_file):
    """Read candidate MPS stations in the Dommel-Warmbeek project area."""
    mps = gpd.read_file(gpkg_file)

    mps['latitude'] = pd.to_numeric(mps['station_latitude'], errors='coerce')
    mps['longitude'] = pd.to_numeric(mps['station_longitude'], errors='coerce')

    # Rough bounding box for the project area.
    mps = mps[
        mps['longitude'].between(5.20, 5.60)
        & mps['latitude'].between(51.10, 51.35)
    ].copy()

    # Convert to Lambert 72 so distances are in metres.
    mps_lambert = mps.to_crs(31370)
    mps['x_mps'] = mps_lambert.geometry.x
    mps['y_mps'] = mps_lambert.geometry.y

    return mps[
        [
            'station_name', 'station_no', 'station_id',
            'longitude', 'latitude', 'x_mps', 'y_mps',
        ]
    ].sort_values('station_name').reset_index(drop=True)


mps_candidates = read_project_mps_candidates(MPS_GPKG_FILE)

print('\nCandidate MPS coordinate points from GPKG:')
print(mps_candidates[['station_name', 'station_no']].to_string(index=False))

## Cell 12 - Create provisional MP01-MP07 coordinate candidates

In [ ]:
mp_measurements = pd.DataFrame([
    {'mps_station_id': 'MP01', 'mps_location_name': 'Goudbergstraat', 'mps_watercourse': 'Dommel'},
    {'mps_station_id': 'MP02', 'mps_location_name': 'Hoksentstraat', 'mps_watercourse': 'Dommel'},
    {'mps_station_id': 'MP03', 'mps_location_name': 'Watermolen van Molhem', 'mps_watercourse': 'Dommel'},
    {'mps_station_id': 'MP04', 'mps_location_name': 'Warmbeek downstream of canal', 'mps_watercourse': 'Warmbeek'},
    {'mps_station_id': 'MP06', 'mps_location_name': 'Warmbeek upstream of Prinsenloop', 'mps_watercourse': 'Warmbeek'},
    {'mps_station_id': 'MP07', 'mps_location_name': 'Eindergatloop', 'mps_watercourse': 'Eindergatloop'},
])

mp_candidate_rows = []

for _, mp in mp_measurements.iterrows():
    candidates = mps_candidates[
        mps_candidates['station_name'].str.contains(
            mp['mps_watercourse'],
            case=False,
            na=False,
        )
    ].copy()

    for _, candidate in candidates.iterrows():
        mp_candidate_rows.append({
            'mps_station_id': mp['mps_station_id'],
            'mps_location_name': mp['mps_location_name'],
            'mps_watercourse': mp['mps_watercourse'],
            'candidate_station_name': candidate['station_name'],
            'candidate_station_no': candidate['station_no'],
            'candidate_longitude': candidate['longitude'],
            'candidate_latitude': candidate['latitude'],
            'x_mps': candidate['x_mps'],
            'y_mps': candidate['y_mps'],
            'coordinate_status': 'candidate_only_exact_2017_measurement_location_not_confirmed',
        })

mp_coordinate_candidates = pd.DataFrame(mp_candidate_rows)

print('\nCandidate coordinate count per MP measurement ID:')
print(mp_coordinate_candidates['mps_station_id'].value_counts().sort_index().to_string())

## Cell 13 - Spatial mapping: CSO point to MP01-MP07 candidates

In [ ]:
mp_distance_rows = []

for _, cso in point_summary.iterrows():
    for _, mp in mp_coordinate_candidates.iterrows():
        dx = cso['x_m'] - mp['x_mps']
        dy = cso['y_m'] - mp['y_mps']
        distance_m = (dx**2 + dy**2) ** 0.5

        mp_distance_rows.append({
            'site': cso['site'],
            'source_name': cso['source_name'],
            'physical_point_id': cso['physical_point_id'],
            'x_cso': cso['x_m'],
            'y_cso': cso['y_m'],
            'receiving_watercourse': cso['receiving_watercourse'],
            'has_any_flow_data': cso['has_any_flow_data'],
            'has_any_emission_2017': cso['has_any_emission_2017'],
            'total_volume_m3_2017': cso['total_volume_m3_2017'],
            'percent_zero_flow_rows': cso['percent_zero_flow_rows'],
            'mps_station_id': mp['mps_station_id'],
            'mps_location_name': mp['mps_location_name'],
            'mps_watercourse': mp['mps_watercourse'],
            'candidate_station_name': mp['candidate_station_name'],
            'candidate_station_no': mp['candidate_station_no'],
            'distance_m': round(distance_m, 1),
            'mapping_status': 'provisional_straight_line_distance_only',
        })

cso_to_mp_all_candidates = pd.DataFrame(mp_distance_rows)

# For each CSO point and MP measurement ID, keep the nearest candidate coordinate.
cso_to_mp_nearest_candidate = (
    cso_to_mp_all_candidates
    .sort_values('distance_m')
    .groupby(['physical_point_id', 'mps_station_id'], as_index=False)
    .first()
    .sort_values(['site', 'source_name', 'mps_station_id'])
    .reset_index(drop=True)
)

print('\nCSO-to-MP candidate distance rows:')
print(len(cso_to_mp_nearest_candidate))

## Cell 14 - Nearest candidate MPS coordinate per CSO point

In [ ]:
nearest_candidate_mps = (
    cso_to_mp_all_candidates
    .sort_values('distance_m')
    .groupby('physical_point_id', as_index=False)
    .first()
    .sort_values(['site', 'source_name'])
    .reset_index(drop=True)
)

nearest_summary = (
    nearest_candidate_mps
    .groupby('site', as_index=False)
    .agg(
        cso_points=('physical_point_id', 'count'),
        median_nearest_distance_m=('distance_m', 'median'),
        max_nearest_distance_m=('distance_m', 'max'),
    )
)

print('\nNearest candidate MPS distance summary:')
print(nearest_summary.round(1).to_string(index=False))

## Cell 15 - Save outputs

In [ ]:
assumptions = pd.DataFrame([
    {
        'topic': 'Available CSO sites',
        'assumption_or_limitation': 'Only Eksel and Overpelt CSO time-series files are available here.',
    },
    {
        'topic': 'Excluded points',
        'assumption_or_limitation': 'RWZI overflow rows in Lozingspunten are excluded from this CSO-only overview.',
    },
    {
        'topic': 'MPS coordinates',
        'assumption_or_limitation': 'MPS_stations.gpkg is treated as a candidate coordinate catalogue, not confirmed MP01-MP07 coordinates.',
    },
    {
        'topic': 'Spatial mapping',
        'assumption_or_limitation': 'Distances are straight-line distances in metres, not river-path distances and not residence times.',
    },
    {
        'topic': 'Routing',
        'assumption_or_limitation': 'No final CSO-to-MPS routing is performed because exact MPS locations and river topology are not confirmed.',
    },
])

point_summary.to_csv(OUTPUT_POINT_SUMMARY, index=False)
cso_to_mp_nearest_candidate.to_csv(OUTPUT_MP_DISTANCE, index=False)

with pd.ExcelWriter(OUTPUT_WORKBOOK, engine='xlsxwriter') as writer:
    site_summary.round(3).to_excel(writer, sheet_name='Site_Summary', index=False)
    point_summary.round(3).to_excel(writer, sheet_name='Point_Summary', index=False)
    incomplete_points.to_excel(writer, sheet_name='Incomplete_Points', index=False)
    mps_candidates.to_excel(writer, sheet_name='MPS_GPKG_Candidates', index=False)
    mp_coordinate_candidates.to_excel(writer, sheet_name='MP_Coordinate_Candidates', index=False)
    nearest_candidate_mps.to_excel(writer, sheet_name='Nearest_MPS_Candidate', index=False)
    cso_to_mp_nearest_candidate.to_excel(writer, sheet_name='CSO_to_MP_Candidates', index=False)
    assumptions.to_excel(writer, sheet_name='Assumptions', index=False)

print('\nSaved outputs:')
print(OUTPUT_WORKBOOK)
print(OUTPUT_POINT_SUMMARY)
print(OUTPUT_MP_DISTANCE)